# 🤖 Model Training — Retail Demand Forecasting

Trains a **SynapseML LightGBM Regressor** to predict daily product demand
per store-product combination.

**Target:** `daily_quantity` (continuous — units sold per day)

**Reads from:** `gold_ml_features`

**Writes to:** `gold_ml_model_metrics`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
features_df = spark.read.table('gold_ml_features')
print(f'Feature rows: {features_df.count():,}')
features_df.select('daily_quantity').describe().show()

In [ ]:
# Numeric features
numeric_features = [
    'transaction_count', 'avg_discount', 'avg_price', 'daily_revenue',
    'day_of_week', 'day_of_month', 'month', 'week_of_year', 'is_weekend',
    'demand_lag_1', 'demand_lag_7', 'revenue_lag_1',
    'unit_cost', 'margin_pct',
]

# Index categoricals
cat_cols = ['category', 'subcategory', 'region', 'store_format']
indexed_df = features_df
cat_idx_cols = []
for c in cat_cols:
    idx_col = f'{c}_idx'
    indexer = StringIndexer(inputCol=c, outputCol=idx_col, handleInvalid='keep')
    indexed_df = indexer.fit(indexed_df).transform(indexed_df)
    cat_idx_cols.append(idx_col)

all_features = numeric_features + cat_idx_cols

assembler = VectorAssembler(inputCols=all_features, outputCol='features', handleInvalid='skip')
model_df = assembler.transform(indexed_df).select('features', col('daily_quantity').alias('label'))
print(f'Model-ready rows: {model_df.count():,}')
print(f'Feature count: {len(all_features)}')

In [ ]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
print(f'Train: {train_df.count():,} rows')
print(f'Test:  {test_df.count():,} rows')

In [ ]:
from synapse.ml.lightgbm import LightGBMRegressor

lgbm = LightGBMRegressor(
    featuresCol='features',
    labelCol='label',
    predictionCol='prediction',
    numLeaves=31,
    numIterations=200,
    learningRate=0.05,
    featureFraction=0.8,
    baggingFraction=0.8,
    baggingFreq=5,
    objective='regression',
    metric='rmse',
)

model = lgbm.fit(train_df)
print('LightGBM regressor trained')

In [ ]:
predictions = model.transform(test_df)

rmse_eval = RegressionEvaluator(labelCol='label', predictionCol='prediction', metricName='rmse')
mae_eval = RegressionEvaluator(labelCol='label', predictionCol='prediction', metricName='mae')
r2_eval = RegressionEvaluator(labelCol='label', predictionCol='prediction', metricName='r2')

rmse = rmse_eval.evaluate(predictions)
mae = mae_eval.evaluate(predictions)
r2 = r2_eval.evaluate(predictions)

print(f'\n=== Model Evaluation ===')
print(f'RMSE: {rmse:.4f}')
print(f'MAE:  {mae:.4f}')
print(f'R²:   {r2:.4f}')

In [ ]:
metrics_data = [
    ('retail-sales', 'demand-forecasting', 'LightGBMRegressor',
     len(all_features), train_df.count(), test_df.count(),
     float(rmse), float(mae), float(r2), 0.0)
]
metrics_df = spark.createDataFrame(
    metrics_data,
    ['demo_id', 'use_case', 'model_type', 'feature_count',
     'train_rows', 'test_rows', 'rmse', 'mae', 'r2', 'placeholder']
).withColumn('trained_at', current_timestamp())

metrics_df.write.mode('overwrite').format('delta').saveAsTable('gold_ml_model_metrics')
print('Model metrics saved to gold_ml_model_metrics')
metrics_df.show(truncate=False)

In [ ]:
model.save('Files/models/demand_forecasting_lgbm')
print('Model saved to Files/models/demand_forecasting_lgbm')